# LeetCode 345 — Reverse Vowels of a String bằng Python

**Link đề:** https://leetcode.com/problems/reverse-vowels-of-a-string/
**Độ khó:** Easy · **Chủ đề:** Two Pointers, String

```python
class Solution:
    def reverseVowels(self, s: str) -> str:
        ...
```

## Mục tiêu học
1. Dùng **`set`** để kiểm tra "ký tự có phải nguyên âm không" trong `O(1)` trung bình, nhớ cả **chữ hoa**.
2. Hiểu vì sao phải đổi chuỗi sang **list** (chuỗi bất biến) rồi `"".join(...)` lại.
3. Đi từ cách trực giác (gom nguyên âm vào **stack** rồi lấy ra ngược) tới **hai con trỏ có bước nhảy qua phần tử không liên quan**.
4. Trace chi tiết khi hai con trỏ "bỏ qua" phụ âm, chứng minh tính đúng bằng invariant.
5. So sánh 4 cách (stack, chỉ số, hai con trỏ, `re.sub`), phân tích Time/Space, tự giải các biến thể "đảo có chọn lọc".

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài

Cho chuỗi `s`. Hãy **chỉ đảo ngược các nguyên âm** trong chuỗi và trả về chuỗi kết quả. Các ký tự khác **giữ nguyên vị trí**.

Nguyên âm là `'a', 'e', 'i', 'o', 'u'`, có thể ở **cả chữ thường và chữ hoa**, xuất hiện nhiều lần.

| Input `s` | Output | Giải thích |
|---|---|---|
| `"IceCreAm"` | `"AceCreIm"` | nguyên âm theo thứ tự: `I, e, e, A` → đảo thành `A, e, e, I` |
| `"leetcode"` | `"leotcede"` | nguyên âm `e, e, o, e` → `e, o, e, e` |

**Constraints:** `1 <= s.length <= 3 · 10^5`; `s` gồm các ký tự ASCII in được.

Câu hỏi đặt ra:
- Chữ `'A'` có phải nguyên âm không? (Có — đề nói cả chữ hoa.)
- `'y'` có phải nguyên âm không? (Không — chỉ 5 chữ trên.)
- Chuỗi không có nguyên âm, hoặc chỉ có 1 nguyên âm thì sao? (Giữ nguyên.)

In [1]:
s = "IceCreAm"
vowel_positions = [(i, ch) for i, ch in enumerate(s) if ch in "aeiouAEIOU"]
print("Vị trí và nguyên âm:", vowel_positions)
print("Các nguyên âm theo thứ tự:", [ch for _, ch in vowel_positions])

Vị trí và nguyên âm: [(0, 'I'), (2, 'e'), (5, 'e'), (6, 'A')]
Các nguyên âm theo thứ tự: ['I', 'e', 'e', 'A']


## 2. Nền tảng Python cho bài này

### 2.1. `set` — tập hợp, kiểm tra `in` nhanh

`set` lưu các phần tử **không trùng**, kiểm tra `x in my_set` trung bình `O(1)` (nhờ băm — hashing). Với chuỗi `"aeiouAEIOU"` chỉ 10 ký tự thì `in` trên chuỗi cũng nhanh, nhưng `set` là thói quen tốt khi tập tra cứu lớn hơn.

In [2]:
VOWELS = set("aeiouAEIOU")
print(VOWELS)                    # thứ tự in ra có thể khác — set không có thứ tự
print("A" in VOWELS, "y" in VOWELS, "e" in VOWELS)
print(len(VOWELS))

{'o', 'u', 'e', 'i', 'I', 'A', 'E', 'O', 'a', 'U'}
True False True
10


### 2.2. `str.lower()` — một cách khác để xử lý chữ hoa

`ch.lower() in "aeiou"` cũng đúng, nhưng phải gọi `lower()` mỗi lần. Dùng set chứa sẵn cả hoa lẫn thường thì gọn hơn.

In [3]:
for ch in "AbE":
    print(ch, "->", ch.lower(), "| nguyên âm?", ch.lower() in "aeiou")

A -> a | nguyên âm? True
b -> b | nguyên âm? False
E -> e | nguyên âm? True


### 2.3. Chuỗi bất biến → `list(s)` để sửa, `"".join(...)` để ghép lại

In [4]:
s = "hello"
chars = list(s)
chars[1], chars[4] = chars[4], chars[1]     # hoán đổi 'e' và 'o'
print(chars)
print("".join(chars))                       # 'holle'
print(s)                                    # chuỗi gốc không đổi

['h', 'o', 'l', 'l', 'e']
holle
hello


### 2.4. Stack bằng list: `append` và `pop`

**Stack** (ngăn xếp) là "vào sau ra trước" (LIFO). Với list: `append(x)` đẩy vào cuối, `pop()` lấy phần tử cuối ra. Đẩy vào theo thứ tự `I, e, e, A` thì lấy ra theo thứ tự `A, e, e, I` — tức là **đảo ngược**.

In [5]:
stack = []
for ch in ["I", "e", "e", "A"]:
    stack.append(ch)
    print("push", ch, "->", stack)
while stack:
    print("pop ->", stack.pop(), "| còn", stack)

push I -> ['I']
push e -> ['I', 'e']
push e -> ['I', 'e', 'e']
push A -> ['I', 'e', 'e', 'A']
pop -> A | còn ['I', 'e', 'e']
pop -> e | còn ['I', 'e']
pop -> e | còn ['I']
pop -> I | còn []


## 3. Cách trực giác — gom nguyên âm vào stack, rồi điền lại

1. Duyệt `s`, đẩy mọi nguyên âm vào `stack`.
2. Duyệt `s` lần nữa: gặp nguyên âm thì lấy `stack.pop()` (nguyên âm **cuối cùng** chưa dùng) thay vào; gặp ký tự khác thì giữ nguyên.

In [6]:
VOWELS = set("aeiouAEIOU")

def reverse_vowels_stack(s: str) -> str:
    stack = [ch for ch in s if ch in VOWELS]
    result = []
    for ch in s:
        if ch in VOWELS:
            result.append(stack.pop())
        else:
            result.append(ch)
    return "".join(result)

print(reverse_vowels_stack("IceCreAm"))
print(reverse_vowels_stack("leetcode"))

AceCreIm
leotcede


## 4. Nút thắt

Cách stack **đúng** và chạy `O(n)`, nhưng:
- Duyệt chuỗi **hai lần**.
- Dùng **thêm bộ nhớ** cho `stack` (tới `O(n)` nếu toàn nguyên âm) **cộng** list kết quả.

Kết quả bắt buộc là một chuỗi mới (`O(n)`), nên không tránh được bộ nhớ cho kết quả. Nhưng có thể bỏ `stack`: **hoán đổi trực tiếp** nguyên âm thứ nhất với nguyên âm cuối, thứ hai với áp chót… — đúng tinh thần bài 344, chỉ thêm việc **nhảy qua phụ âm**.

## 5. Insight — hai con trỏ, mỗi con trỏ dừng ở nguyên âm

Đảo các nguyên âm = hoán đổi **nguyên âm thứ k từ trái** với **nguyên âm thứ k từ phải**.

```text
left  -> tiến sang phải cho tới khi gặp nguyên âm
right -> lùi sang trái cho tới khi gặp nguyên âm
nếu left < right: hoán đổi, rồi cả hai bước tiếp
```

Ví dụ `"IceCreAm"`:
```text
I c e C r e A m
^           ^       left=0 ('I'), right=6 ('A') -> đổi -> A c e C r e I m
    ^     ^         left=2 ('e'), right=5 ('e') -> đổi (giống nhau)
          gặp nhau  -> dừng
```

## 6. Lời giải chính — giảng từng đoạn rồi ghép

**Đoạn 1 — chuẩn bị:**
```python
chars = list(s)
left, right = 0, len(chars) - 1
```

**Đoạn 2 — vòng ngoài khi hai con trỏ chưa gặp nhau:**
```python
while left < right:
```

**Đoạn 3 — hai vòng trong "nhảy qua phụ âm", luôn giữ `left < right`:**
```python
    while left < right and chars[left] not in VOWELS:
        left += 1
    while left < right and chars[right] not in VOWELS:
        right -= 1
```
Điều kiện `left < right` trong vòng trong ngăn con trỏ **chạy vượt** ra ngoài khi không còn nguyên âm.

**Đoạn 4 — hoán đổi và bước tiếp:**
```python
    chars[left], chars[right] = chars[right], chars[left]
    left += 1
    right -= 1
```

**Đoạn 5 — ghép lại:** `return "".join(chars)`.

In [7]:
def reverse_vowels_two_pointers(s: str) -> str:
    chars = list(s)
    left, right = 0, len(chars) - 1
    while left < right:
        while left < right and chars[left] not in VOWELS:
            left += 1
        while left < right and chars[right] not in VOWELS:
            right -= 1
        chars[left], chars[right] = chars[right], chars[left]
        left += 1
        right -= 1
    return "".join(chars)

for s in ["IceCreAm", "leetcode", "bcd", "a", "aA", "hello world"]:
    print(f"{s!r:>14} -> {reverse_vowels_two_pointers(s)!r}")

    'IceCreAm' -> 'AceCreIm'
    'leetcode' -> 'leotcede'
         'bcd' -> 'bcd'
           'a' -> 'a'
          'aA' -> 'Aa'
 'hello world' -> 'hollo werld'


**Câu hỏi:** Khi hai vòng trong dừng vì `left == right` (không còn nguyên âm ở giữa), đoạn 4 vẫn hoán đổi `chars[left]` với chính nó — có sai không? **Không sai**: hoán đổi một phần tử với chính nó không đổi gì. Sau đó `left > right` và vòng ngoài dừng.

## 7. TRACE từng bước

Bảng tay cho `s = "leetcode"` (chỉ số: `l0 e1 e2 t3 c4 o5 d6 e7`):

| Vòng | left bắt đầu → dừng ở | right bắt đầu → dừng ở | Hoán đổi | chars sau |
|---:|---|---|---|---|
| 1 | 0 (`l`) → 1 (`e`) | 7 (`e`) → 7 | e₁ ↔ e₇ | l e e t c o d e |
| 2 | 2 (`e`) → 2 | 6 (`d`) → 5 (`o`) | e₂ ↔ o₅ | l e o t c e d e |
| 3 | 3 (`t`) → 4 → 4 (left == right) | 4 | (với chính nó) | không đổi |
| — | left = 5, right = 3 | | dừng | **`leotcede`** |

In [8]:
def trace_reverse_vowels(s: str) -> str:
    chars = list(s)
    left, right = 0, len(chars) - 1
    step = 1
    print("bắt đầu:", "".join(chars))
    while left < right:
        l0, r0 = left, right
        while left < right and chars[left] not in VOWELS:
            left += 1
        while left < right and chars[right] not in VOWELS:
            right -= 1
        note = "hoán đổi" if left < right else "left == right, hoán đổi với chính nó"
        a, c = chars[left], chars[right]
        chars[left], chars[right] = chars[right], chars[left]
        print(f"vòng {step}: left {l0}->{left} ({a!r}), right {r0}->{right} ({c!r}) | {note} -> {''.join(chars)}")
        left += 1
        right -= 1
        step += 1
    print(f"dừng: left={left}, right={right} -> {''.join(chars)!r}")
    return "".join(chars)

trace_reverse_vowels("leetcode")
print()
trace_reverse_vowels("IceCreAm")

bắt đầu: leetcode
vòng 1: left 0->1 ('e'), right 7->7 ('e') | hoán đổi -> leetcode
vòng 2: left 2->2 ('e'), right 6->5 ('o') | hoán đổi -> leotcede
vòng 3: left 3->4 ('c'), right 4->4 ('c') | left == right, hoán đổi với chính nó -> leotcede
dừng: left=5, right=3 -> 'leotcede'

bắt đầu: IceCreAm
vòng 1: left 0->0 ('I'), right 7->6 ('A') | hoán đổi -> AceCreIm
vòng 2: left 1->2 ('e'), right 5->5 ('e') | hoán đổi -> AceCreIm
vòng 3: left 3->4 ('r'), right 4->4 ('r') | left == right, hoán đổi với chính nó -> AceCreIm
dừng: left=5, right=3 -> 'AceCreIm'


'AceCreIm'

In [9]:
trace_reverse_vowels("xyz")          # không có nguyên âm: con trỏ trái chạy tới gặp con trỏ phải
print()
trace_reverse_vowels("Programming")  # chữ hoa P không phải nguyên âm

bắt đầu: xyz
vòng 1: left 0->2 ('z'), right 2->2 ('z') | left == right, hoán đổi với chính nó -> xyz
dừng: left=3, right=1 -> 'xyz'

bắt đầu: Programming
vòng 1: left 0->2 ('o'), right 10->8 ('i') | hoán đổi -> Prigrammong
vòng 2: left 3->5 ('a'), right 7->5 ('a') | left == right, hoán đổi với chính nó -> Prigrammong
dừng: left=6, right=4 -> 'Prigrammong'


'Prigrammong'

## 8. Các lời giải khác

### Cách 3 — gom **chỉ số** của nguyên âm, rồi hoán đổi theo cặp
Rõ ràng, dễ debug: `idx = [i for i, ch in enumerate(s) if ch in VOWELS]`, rồi hoán đổi `idx[k]` với `idx[-1-k]`.

In [10]:
def reverse_vowels_indices(s: str) -> str:
    chars = list(s)
    idx = [i for i, ch in enumerate(chars) if ch in VOWELS]
    for k in range(len(idx) // 2):
        i, j = idx[k], idx[-1 - k]
        chars[i], chars[j] = chars[j], chars[i]
    return "".join(chars)

print(reverse_vowels_indices("IceCreAm"), reverse_vowels_indices("leetcode"))

AceCreIm leotcede


### Cách 4 — `re.sub` với hàm thay thế (sáng tạo, thư viện chuẩn)

Module `re` (biểu thức chính quy) tìm mọi ký tự khớp `[aeiouAEIOU]`. `re.sub(pattern, func, s)` gọi `func(match)` cho **mỗi** lần khớp theo thứ tự trái → phải và thay bằng giá trị trả về. Ta cho `func` lấy nguyên âm từ **cuối** danh sách (stack).

In [11]:
import re

def reverse_vowels_regex(s: str) -> str:
    stack = re.findall(r"[aeiouAEIOU]", s)     # list các nguyên âm theo thứ tự
    return re.sub(r"[aeiouAEIOU]", lambda m: stack.pop(), s)

print(re.findall(r"[aeiouAEIOU]", "IceCreAm"))
print(reverse_vowels_regex("IceCreAm"), reverse_vowels_regex("leetcode"))

['I', 'e', 'e', 'A']
AceCreIm leotcede


## 9. Code nộp LeetCode

### Cách chính — hai con trỏ

In [12]:
class Solution:
    def reverseVowels(self, s: str) -> str:
        vowels = set("aeiouAEIOU")
        chars = list(s)
        left, right = 0, len(chars) - 1
        while left < right:
            while left < right and chars[left] not in vowels:
                left += 1
            while left < right and chars[right] not in vowels:
                right -= 1
            chars[left], chars[right] = chars[right], chars[left]
            left += 1
            right -= 1
        return "".join(chars)

print(Solution().reverseVowels("IceCreAm"), Solution().reverseVowels("leetcode"))

AceCreIm leotcede


### Cách nộp thay thế — stack (ngắn gọn)

In [13]:
class SolutionStack:
    def reverseVowels(self, s: str) -> str:
        vowels = set("aeiouAEIOU")
        stack = [ch for ch in s if ch in vowels]
        return "".join(stack.pop() if ch in vowels else ch for ch in s)

print(SolutionStack().reverseVowels("IceCreAm"))

AceCreIm


## 10. Vì sao đúng?

Gọi `v₁, v₂, …, v_m` là vị trí các nguyên âm theo thứ tự tăng dần. Kết quả mong muốn: ký tự tại `v_k` đổi chỗ với ký tự tại `v_{m+1-k}`.

**Invariant (đầu mỗi vòng ngoài):** tồn tại `k` sao cho
- các cặp `(v₁, v_m), …, (v_k, v_{m+1-k})` **đã hoán đổi xong**;
- `left` nằm ngay sau `v_k` (≤ `v_{k+1}`), `right` nằm ngay trước `v_{m+1-k}` (≥ `v_{m-k}`);
- đoạn `chars[left..right]` chưa bị sửa.

**Mỗi vòng:** vòng trong của `left` dừng tại nguyên âm đầu tiên `≥ left`, tức là `v_{k+1}`; vòng trong của `right` dừng tại `v_{m-k}`. Nếu `left < right`, hoán đổi đúng cặp kế tiếp ✔. Nếu `left == right`, không còn cặp nào (còn 0 hoặc 1 nguyên âm ở giữa) và hoán đổi với chính nó vô hại ✔.

**Không bỏ sót:** con trỏ chỉ nhảy qua **phụ âm** — không bao giờ nhảy qua nguyên âm chưa xử lý. **Phụ âm không bị di chuyển:** chỉ hoán đổi khi cả hai đầu là nguyên âm (hoặc `left == right`).

## 11. Time / Space Complexity và so sánh

Gọi `n = len(s)`, `m` = số nguyên âm.

| Cách | Time | Bộ nhớ phụ (ngoài kết quả) | Ghi chú |
|---|---|---|---|
| Stack | `O(n)` (2 lượt) | `O(m)` | dễ nghĩ nhất |
| Danh sách chỉ số | `O(n)` | `O(m)` | dễ debug |
| **Hai con trỏ** | **`O(n)`** (1 lượt, mỗi con trỏ đi tối đa n bước) | **`O(1)`** ngoài `chars` | chuẩn phỏng vấn |
| `re.sub` | `O(n)` | `O(m)` | ngắn, ít dùng khi phỏng vấn |

Vì chuỗi bất biến, mọi cách đều cần `O(n)` để tạo chuỗi kết quả (`list(s)` hoặc list kết quả).

**Tại sao hai con trỏ vẫn `O(n)` dù có vòng `while` lồng nhau?** Vì `left` chỉ tăng, `right` chỉ giảm, và chúng không vượt qua nhau → tổng số bước của cả hai ≤ `n`.

## 12. Bộ test — mẫu đề, biên, đối chiếu

In [14]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual!r}")

sol = Solution()
check("đề 1", sol.reverseVowels("IceCreAm"), "AceCreIm")
check("đề 2", sol.reverseVowels("leetcode"), "leotcede")
check("1 ký tự nguyên âm", sol.reverseVowels("a"), "a")
check("1 ký tự phụ âm", sol.reverseVowels("z"), "z")
check("không nguyên âm", sol.reverseVowels("bcdfg"), "bcdfg")
check("toàn nguyên âm", sol.reverseVowels("aeiou"), "uoiea")
check("hoa + thường", sol.reverseVowels("aA"), "Aa")
check("y không phải nguyên âm", sol.reverseVowels("yay"), "yay")
check("ký tự đặc biệt", sol.reverseVowels("a.b,c!E"), "E.b,c!a")
check("1 nguyên âm giữa phụ âm", sol.reverseVowels("xxaxx"), "xxaxx")
big = "ab" * 150000
assert sol.reverseVowels(big) == big        # mọi nguyên âm đều là 'a'
print("PASS n = 3·10^5")

PASS đề 1: 'AceCreIm'
PASS đề 2: 'leotcede'
PASS 1 ký tự nguyên âm: 'a'
PASS 1 ký tự phụ âm: 'z'
PASS không nguyên âm: 'bcdfg'
PASS toàn nguyên âm: 'uoiea'
PASS hoa + thường: 'Aa'
PASS y không phải nguyên âm: 'yay'
PASS ký tự đặc biệt: 'E.b,c!a'
PASS 1 nguyên âm giữa phụ âm: 'xxaxx'
PASS n = 3·10^5


In [15]:
import random
random.seed(345)
alphabet = "aeiouAEIOUbcdxyzBCDXYZ .,!1"
methods = [reverse_vowels_stack, reverse_vowels_two_pointers, reverse_vowels_indices,
           reverse_vowels_regex, Solution().reverseVowels, SolutionStack().reverseVowels]
for _ in range(1000):
    s = "".join(random.choice(alphabet) for _ in range(random.randint(1, 25)))
    expected = reverse_vowels_stack(s)
    for f in methods:
        r = f(s)
        assert r == expected, (f, s, r)
    # tính chất: phụ âm giữ nguyên vị trí, chuỗi nguyên âm bị đảo
    assert all((ch in VOWELS) == (r_ch in VOWELS) for ch, r_ch in zip(s, expected))
    assert [c for c in expected if c in VOWELS] == [c for c in s if c in VOWELS][::-1]
print("1000 chuỗi ngẫu nhiên × 6 cách: đều khớp và thỏa các tính chất")

1000 chuỗi ngẫu nhiên × 6 cách: đều khớp và thỏa các tính chất


## 13. Lỗi người mới hay gặp

### Lỗi 1: Quên chữ hoa
`VOWELS = set("aeiou")` → `"IceCreAm"` sai.

In [16]:
def reverse_vowels_lower_only(s):
    vowels = set("aeiou")                         # SAI: thiếu chữ hoa
    chars = list(s)
    l, r = 0, len(chars) - 1
    while l < r:
        while l < r and chars[l] not in vowels: l += 1
        while l < r and chars[r] not in vowels: r -= 1
        chars[l], chars[r] = chars[r], chars[l]
        l += 1; r -= 1
    return "".join(chars)

print("sai :", reverse_vowels_lower_only("IceCreAm"), "| đúng:", Solution().reverseVowels("IceCreAm"))

sai : IceCreAm | đúng: AceCreIm


### Lỗi 2: Vòng trong thiếu điều kiện `left < right`
Với chuỗi không có nguyên âm, `left` chạy tới `len(s)` → `IndexError`.

### Lỗi 3: Dùng `if` thay cho `while` để bỏ qua phụ âm
Chỉ bỏ qua **một** phụ âm mỗi vòng, rồi vẫn hoán đổi → phụ âm bị di chuyển.

In [17]:
def reverse_vowels_no_bound(s):
    chars = list(s)
    l, r = 0, len(chars) - 1
    while l < r:
        while chars[l] not in VOWELS: l += 1     # SAI: thiếu l < r
        while chars[r] not in VOWELS: r -= 1
        chars[l], chars[r] = chars[r], chars[l]
        l += 1; r -= 1
    return "".join(chars)

try:
    reverse_vowels_no_bound("xyz")
except IndexError as e:
    print("IndexError:", e)

def reverse_vowels_if(s):
    chars = list(s)
    l, r = 0, len(chars) - 1
    while l < r:
        if chars[l] not in VOWELS: l += 1        # SAI: chỉ bỏ qua 1 ký tự
        if chars[r] not in VOWELS: r -= 1
        chars[l], chars[r] = chars[r], chars[l]
        l += 1; r -= 1
    return "".join(chars)

print("sai :", reverse_vowels_if("leetcode"), "| đúng:", Solution().reverseVowels("leetcode"))

IndexError: list index out of range
sai : leoctede | đúng: leotcede


### Lỗi 4: Sửa trực tiếp chuỗi `s[left] = ...` → `TypeError`.

### Lỗi 5: Ghép chuỗi bằng `+=` trong vòng lặp
`result += ch` tạo chuỗi mới mỗi lần → có thể `O(n²)` trong trường hợp xấu. Dùng list + `"".join` an toàn hơn.

In [18]:
try:
    t = "hello"
    t[1] = "o"
except TypeError as e:
    print("TypeError:", e)

TypeError: 'str' object does not support item assignment


## 14. Mẹo nhận dạng pattern

- **"Đảo ngược nhưng chỉ một loại phần tử, phần còn lại giữ nguyên"** → hai con trỏ + vòng `while` **nhảy qua** phần tử không liên quan (luôn kèm `left < right`).
- Hoặc: lấy các phần tử cần đảo ra (stack / list chỉ số), xử lý, rồi đặt lại.
- Tập ký tự cần nhận diện → đặt vào `set` một lần ở ngoài vòng lặp.
- Bài cùng khuôn: Reverse Only Letters (917), Valid Palindrome (125: bỏ qua ký tự không phải chữ/số), Sort Array By Parity (905).

## 15. Biến thể

### Biến thể 1 — Reverse Only Letters (LeetCode 917)
Chỉ đảo **chữ cái**, giữ nguyên vị trí các ký tự khác. Thay điều kiện bằng `ch.isalpha()`.

### Biến thể 2 — tham số hóa: đảo mọi ký tự thỏa một hàm điều kiện `keep(ch)`

In [19]:
def reverse_selected(s: str, should_reverse) -> str:
    chars = list(s)
    left, right = 0, len(chars) - 1
    while left < right:
        while left < right and not should_reverse(chars[left]):
            left += 1
        while left < right and not should_reverse(chars[right]):
            right -= 1
        chars[left], chars[right] = chars[right], chars[left]
        left += 1
        right -= 1
    return "".join(chars)

print(reverse_selected("a-bC-dEf-ghIj", str.isalpha))         # Reverse Only Letters: j-Ih-gfE-dCba
print(reverse_selected("a1b2c3", str.isdigit))                # chỉ đảo chữ số: a3b2c1
print(reverse_selected("IceCreAm", lambda c: c in VOWELS))    # quay lại bài gốc

j-Ih-gfE-dCba
a3b2c1
AceCreIm


## 16. Bài tập tự luyện (TODO — không chạy sẵn)

### Bài 1 — Tính tay
Trace hai con trỏ cho `s = "Aa bE"`: ghi left/right dừng ở đâu mỗi vòng, kết quả cuối?

### Bài 2 — Dự đoán output
`"".join(sorted(set("banana")))`, `"Y" in set("aeiouAEIOU")`, `list("ab") + ["c"]`.

### Bài 3 — Điền chỗ trống

In [20]:
def exercise_reverse_vowels(s: str) -> str:
    vowels = set("aeiouAEIOU")
    chars = list(s)
    left, right = 0, len(chars) - 1
    while left < right:
        # TODO 1: đưa left tới nguyên âm (giữ left < right)
        # TODO 2: đưa right tới nguyên âm (giữ left < right)
        # TODO 3: hoán đổi, tiến hai con trỏ
        pass
    return "".join(chars)

# assert exercise_reverse_vowels("IceCreAm") == "AceCreIm"
# assert exercise_reverse_vowels("leetcode") == "leotcede"

### Bài 4 — Tự viết: đếm nguyên âm và trả về nguyên âm xuất hiện nhiều nhất
Trả về tuple `(tổng số nguyên âm, nguyên âm nhiều nhất viết thường)`; không phân biệt hoa thường; hòa thì chọn chữ nhỏ hơn theo bảng chữ cái; chuỗi không có nguyên âm trả `(0, "")`.

In [21]:
def exercise_vowel_stats(s: str) -> tuple:
    # TODO
    pass

# assert exercise_vowel_stats("IceCreAm") == (4, "e")
# assert exercise_vowel_stats("xyz") == (0, "")
# assert exercise_vowel_stats("Aeae") == (4, "a")

### Bài 5 — Biến thể: Reverse Only Letters (LeetCode 917)

In [22]:
def exercise_reverse_only_letters(s: str) -> str:
    # TODO: hai con trỏ với ch.isalpha()
    pass

# assert exercise_reverse_only_letters("ab-cd") == "dc-ba"
# assert exercise_reverse_only_letters("a-bC-dEf-ghIj") == "j-Ih-gfE-dCba"
# assert exercise_reverse_only_letters("Test1ng-Leet=code-Q!") == "Qedo1ct-eeLg=ntse-T!"

### Bài 6 — Mini challenge: xoay vòng nguyên âm
Thay vì đảo, hãy **xoay** các nguyên âm sang phải 1 bước: nguyên âm cuối lên vị trí nguyên âm đầu, các nguyên âm khác dời sang vị trí nguyên âm kế tiếp. Ví dụ `"leetcode"` (nguyên âm `e,e,o,e`) → nguyên âm mới `e,e,e,o` → `"leetcedo"`.

In [23]:
def exercise_rotate_vowels(s: str) -> str:
    # TODO: lấy list chỉ số nguyên âm, xoay list giá trị, đặt lại
    pass

# assert exercise_rotate_vowels("leetcode") == "leetcedo"
# assert exercise_rotate_vowels("IceCreAm") == "AcICreem"
# assert exercise_rotate_vowels("bcd") == "bcd"

## 17. Gợi ý 3 mức

**Bài 1** — (1) Chỉ số: `A0 a1 ' '2 b3 E4`. (2) Vòng 1: left=0, right=4. (3) Vòng 2: left=1, right dừng ở 1 vì gặp left.

**Bài 2** — (1) `set` bỏ trùng. (2) `sorted` trả về list đã sắp xếp. (3) `Y` không nằm trong 10 ký tự.

**Bài 3** — (1) Hai vòng `while ... not in vowels`. (2) Mỗi vòng trong đều có `left < right`. (3) Xem mục 6.

**Bài 4** — (1) Đếm bằng dict với key `ch.lower()`. (2) `max(counts, key=lambda v: (counts[v], -ord(v)))` hoặc duyệt "aeiou" theo thứ tự và chỉ cập nhật khi lớn hơn hẳn. (3) Xử lý trường hợp dict rỗng.

**Bài 5** — (1) Thay `in vowels` bằng `.isalpha()`. (2) Có thể dùng `reverse_selected`. (3) Nhưng hãy tự viết lại vòng lặp.

**Bài 6** — (1) `idx = [i for i, ch in enumerate(s) if ch in vowels]`. (2) `vals = [s[i] for i in idx]`, xoay phải: `vals[-1:] + vals[:-1]`. (3) Gán `chars[idx[k]] = new_vals[k]`.

## 18. Đáp án tham khảo

**Bài 1:** vòng 1: `left=0 ('A')`, `right=4 ('E')` → hoán đổi → `"Ea bA"`; vòng 2: `left=1 ('a')`, right lùi 3 → 2 → 1 (dừng vì gặp left) → hoán đổi với chính nó. Kết quả `"Ea bA"`.

**Bài 2:** `'abn'`, `False`, `['a', 'b', 'c']`.

In [24]:
def answer_reverse_vowels(s: str) -> str:
    vowels = set("aeiouAEIOU")
    chars = list(s)
    left, right = 0, len(chars) - 1
    while left < right:
        while left < right and chars[left] not in vowels:
            left += 1
        while left < right and chars[right] not in vowels:
            right -= 1
        chars[left], chars[right] = chars[right], chars[left]
        left += 1
        right -= 1
    return "".join(chars)

def answer_vowel_stats(s: str) -> tuple:
    counts = {}
    for ch in s:
        low = ch.lower()
        if low in "aeiou":
            counts[low] = counts.get(low, 0) + 1
    if not counts:
        return (0, "")
    best = ""
    for v in "aeiou":                       # duyệt theo bảng chữ cái -> hòa thì giữ chữ nhỏ hơn
        if v in counts and (best == "" or counts[v] > counts[best]):
            best = v
    return (sum(counts.values()), best)

def answer_reverse_only_letters(s: str) -> str:
    chars = list(s)
    left, right = 0, len(chars) - 1
    while left < right:
        while left < right and not chars[left].isalpha():
            left += 1
        while left < right and not chars[right].isalpha():
            right -= 1
        chars[left], chars[right] = chars[right], chars[left]
        left += 1
        right -= 1
    return "".join(chars)

def answer_rotate_vowels(s: str) -> str:
    vowels = set("aeiouAEIOU")
    chars = list(s)
    idx = [i for i, ch in enumerate(chars) if ch in vowels]
    vals = [chars[i] for i in idx]
    new_vals = vals[-1:] + vals[:-1]
    for k, i in enumerate(idx):
        chars[i] = new_vals[k]
    return "".join(chars)

assert answer_reverse_vowels("Aa bE") == "Ea bA"
assert ("".join(sorted(set("banana"))), "Y" in set("aeiouAEIOU"), list("ab") + ["c"]) == ("abn", False, ["a", "b", "c"])
assert answer_reverse_vowels("IceCreAm") == "AceCreIm" and answer_reverse_vowels("leetcode") == "leotcede"
assert answer_vowel_stats("IceCreAm") == (4, "e") and answer_vowel_stats("xyz") == (0, "") and answer_vowel_stats("Aeae") == (4, "a")
assert answer_reverse_only_letters("ab-cd") == "dc-ba"
assert answer_reverse_only_letters("a-bC-dEf-ghIj") == "j-Ih-gfE-dCba"
assert answer_reverse_only_letters("Test1ng-Leet=code-Q!") == "Qedo1ct-eeLg=ntse-T!"
assert answer_rotate_vowels("leetcode") == "leetcedo"
assert answer_rotate_vowels("IceCreAm") == "AcICreem" and answer_rotate_vowels("bcd") == "bcd"
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 19. Template nhớ nhanh + checklist submit

```python
class Solution:
    def reverseVowels(self, s: str) -> str:
        vowels = set("aeiouAEIOU")
        chars = list(s)
        left, right = 0, len(chars) - 1
        while left < right:
            while left < right and chars[left] not in vowels:
                left += 1
            while left < right and chars[right] not in vowels:
                right -= 1
            chars[left], chars[right] = chars[right], chars[left]
            left += 1
            right -= 1
        return "".join(chars)
```

**Template "đảo có chọn lọc":** hai con trỏ ngược chiều → mỗi con trỏ `while` nhảy qua phần tử không cần đảo (kèm `left < right`) → hoán đổi → tiến cả hai.

**Checklist trước khi submit:**
- [ ] Tập nguyên âm có cả **chữ hoa**?
- [ ] Đổi sang `list` trước khi hoán đổi, `"".join` khi trả về?
- [ ] Vòng trong dùng `while` (không phải `if`) và có `left < right`?
- [ ] Thử chuỗi không có nguyên âm, 1 ký tự, toàn nguyên âm?
- [ ] Trả về **chuỗi** (đề yêu cầu `str`, khác bài 344)?